# Multi-layer perceptron: intuition
Two perceptrons, one perceptron that combines them, and small MLPs on non-linear data.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# the sigmoid squeezes any number into the range 0 to 1
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

## Combining two probabilities

In [ ]:
p1, p2 = 0.7, 0.8
# plain sum, then sigmoid
print(round(sigmoid(p1 + p2), 3))            # 0.818
# weighted sum plus bias, then sigmoid
z = 10 * p1 + 5 * p2 + 3
print(z, sigmoid(z))                          # 14.0, 0.99999917

## The combination that bends the boundary
Perceptron 1: $\sigma(3x_1 + 3x_2)$, perceptron 2: $\sigma(3x_1 - 3x_2)$, combiner: $\sigma(8p_1 + 8p_2 - 12)$.

In [ ]:
def p1(x1, x2): return sigmoid(3 * x1 + 3 * x2)
def p2(x1, x2): return sigmoid(3 * x1 - 3 * x2)
def combined(x1, x2): return sigmoid(8 * p1(x1, x2) + 8 * p2(x1, x2) - 12)

# (0, 0) lies on both lines: each perceptron says 0.5, the combination says 0.018
for pt in [(1, 0), (0, 0)]:
    print(pt, round(p1(*pt), 3), round(p2(*pt), 3), round(combined(*pt), 3))

In [ ]:
# probability maps: drag the slider to move the combiner's bias and watch the boundary change
xs = np.linspace(-3, 3, 151)
X1, X2 = np.meshgrid(xs, xs)
fig = go.Figure()
biases = [-15, -12, -9, -6, -3]
for b in biases:
    Z = sigmoid(8 * p1(X1, X2) + 8 * p2(X1, X2) + b)
    fig.add_trace(go.Contour(x=xs, y=xs, z=Z, zmin=0, zmax=1, visible=(b == -12),
                             colorscale=[[0, "#F6C9C9"], [0.5, "#FFFFFF"], [1, "#CBE5C5"]],
                             contours=dict(start=0.1, end=0.9, size=0.1)))
steps = [dict(method="update", label=str(b), args=[{"visible": [c == b for c in biases]}]) for b in biases]
fig.update_layout(sliders=[dict(active=1, steps=steps, currentvalue=dict(prefix="combiner bias "))],
                  width=600, height=600, template="simple_white")
fig.show()

## The 2-2-1 network of Figure 2

In [ ]:
def mlp_221(cgpa, iq):
    h1 = sigmoid(2 * cgpa + 3 * iq + 6)     # perceptron 1
    h2 = sigmoid(5 * cgpa + 4 * iq + 3)     # perceptron 2
    return sigmoid(10 * h1 + 5 * h2 + 3)   # perceptron 3 combines them

# with these made-up positive weights every student gets almost 1:
# the numbers only show the structure, they are not trained
print(mlp_221(7.5, 110))

## TensorFlow Playground demos with scikit-learn
The same data and models as Figure 4 (code in `images/playground.py`).

In [ ]:
import sys
sys.path.insert(0, "images")
from playground import fit_all

runs = fit_all()
for title, X, y, clf in runs:
    print(f"{title:35s} accuracy {clf.score(X, y):.2f}")

In [ ]:
# decision boundaries, one panel per model
fig = make_subplots(2, 3, subplot_titles=[t for t, *_ in runs])
g = np.linspace(-4, 4, 150)
G1, G2 = np.meshgrid(g, g)
for k, (title, X, y, clf) in enumerate(runs):
    r, c = k // 3 + 1, k % 3 + 1
    Z = clf.predict_proba(np.c_[G1.ravel(), G2.ravel()])[:, 1].reshape(G1.shape)
    fig.add_trace(go.Contour(x=g, y=g, z=Z, showscale=False,
                             colorscale=[[0, "#F6C9C9"], [1, "#CBE5C5"]]), r, c)
    fig.add_trace(go.Scatter(x=X[:, 0], y=X[:, 1], mode="markers", showlegend=False,
                             marker=dict(color=np.where(y == 1, "#54A24B", "#E45756"), size=4)), r, c)
fig.update_layout(width=1000, height=700, template="simple_white")
fig.show()

## Try it
Change `hidden_layer_sizes` or `activation` and refit.

In [ ]:
from sklearn.neural_network import MLPClassifier
X, y = runs[0][1], runs[0][2]                  # the XOR data
for h in [(2,), (3,), (4,), (8,)]:
    clf = MLPClassifier(hidden_layer_sizes=h, activation="logistic", solver="lbfgs",
                        max_iter=5000, random_state=0).fit(X, y)
    print(h, round(clf.score(X, y), 3))